In [1]:
import time
from litellm import completion, completion_cost

def classify_task(user_query: str) -> str:
    """Cheap classifier — uses the fastest free-tier model to decide routing."""
    cls = completion(
        model="groq/llama-3.1-8b-instant",
        messages=[{
            "role": "user",
            "content": (
                f"Classify the following query into EXACTLY one word: "
                f"'code', 'summary', or 'general'. Query: {user_query}\n\nAnswer:"
            )
        }],
        max_tokens=5
    )
    return cls.choices[0].message.content.strip().lower()


def call_with_fallbacks(model_chain, messages):
    """Try each model in order; return the first one that succeeds."""
    last_error = None
    for model in model_chain:
        try:
            return completion(model=model, messages=messages)
        except Exception as e:
            print(f"   ⚠️  {model} failed ({type(e).__name__}), trying next...")
            last_error = e
            continue
    raise last_error


def smart_chat(user_query: str):
    """Routes to the right model based on task type, with fallbacks."""
    task = classify_task(user_query)

    # Groq + Gemini free-tier models only
    routing = {
        "code":    ["groq/llama-3.3-70b-versatile", "gemini/gemini-2.5-flash"],
        "summary": ["gemini/gemini-2.5-flash-lite",  "groq/llama-3.1-8b-instant"],
        "general": ["groq/llama-3.1-8b-instant",     "gemini/gemini-2.5-flash-lite"],
    }
    model_chain = routing.get(task, routing["general"])

    start = time.time()
    response = call_with_fallbacks(
        model_chain=model_chain,
        messages=[{"role": "user", "content": user_query}]
    )
    latency = time.time() - start

    try:
        cost = completion_cost(completion_response=response)
        cost_str = f"${cost:.6f}"
    except Exception:
        cost_str = "n/a"

    return {
        "detected_task": task,
        "model_used":    response.model,
        "answer":        response.choices[0].message.content,
        "latency_sec":   round(latency, 2),
        "cost_usd":      cost_str
    }


# Try it on three very different queries
queries = [
    "Write a Python function to compute Fibonacci numbers.",
    "Summarize the importance of attention mechanism in 2 sentences.",
    "Tell me a fun fact about elephants."
]

for q in queries:
    print("=" * 70)
    print("❓ Q:", q)
    result = smart_chat(q)
    print(f"🏷️  Task:    {result['detected_task']}")
    print(f"🤖 Model:    {result['model_used']}")
    print(f"⏱️  Latency: {result['latency_sec']}s")
    print(f"💰 Cost:    {result['cost_usd']}")
    print(f"💬 Answer:  {result['answer'][:200]}...")

❓ Q: Write a Python function to compute Fibonacci numbers.

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers

🏷️  Task:    code
🤖 Model:    llama-3.3-70b-versatile
⏱️  Latency: 1.21s
💰 Cost:    n/a
💬 Answer:  **Fibonacci Function in Python**

Here's a simple recursive and iterative implementation of a Fibonacci function in Python.

### Recursive Implementation

```python
de...
❓ Q: Summarize the importance of attention mechanism in 2 sentences.

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.

   ⚠️  gemini/gemini-2.5-flash-lite failed (NotFoundError), trying next...

Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.ai/docs/providers


Provider List: https://docs.litellm.a